# Controlled Density Lab — Colab quickstart

Minimal entrypoint. All business logic lives in the package and CLI scripts;
this notebook only orchestrates. These are numerical controlled benchmarks
(D0-D5), **not** SHiP physics: no background rate or FairShip speed-up is
claimed.

Steps: clone → install → device info → tests → run `smoke_v0` → (optional)
persist to Drive after writing locally → demonstrate resume → build report.

Do not hardcode a GPU model. Do not store credentials here. Drive I/O is not
suitable for training batches — write locally first, then copy.

In [ ]:
# 1. Clone the repository (skip if already present).
import os
REPO = 'ship_adaptive_muon_bg'
if not os.path.isdir(REPO):
    !git clone https://github.com/fbientrigo/ship_adaptive_muon_bg.git
%cd $REPO

In [ ]:
# 2. Install the optional stack (torch is CPU-only here; a GPU runtime uses CUDA).
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu
!pip -q install -e '.[dev,flow,lab,tracking]'

In [ ]:
# 3. Display torch / CUDA / device information (no hardcoded GPU model).
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda runtime:', torch.version.cuda)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('selected device:', device)

In [ ]:
# 4. Run a fast test subset (flow + lab invariants) on CPU.
!python -m pytest -q tests/test_affine_coupling_flow.py tests/test_density_metrics.py

In [ ]:
# 5. Run the smoke_v0 campaign (CPU by default; pass --device cuda on a GPU runtime).
!python scripts/run_density_lab.py --config configs/density_lab/smoke_v0.json --device $device

In [ ]:
# 6. (Optional) Persist artifacts to Google Drive AFTER writing locally.
#    Skip this cell if you are not on Colab or do not want Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/density_lab_artifacts
    !cp -r artifacts/density_lab/smoke_v0 /content/drive/MyDrive/density_lab_artifacts/
    print('copied local artifacts to Drive')
except Exception as exc:
    print('Drive step skipped:', exc)

In [ ]:
# 7. Demonstrate resume: re-running skips completed identical run hashes.
!python scripts/run_density_lab.py --config configs/density_lab/smoke_v0.json --device $device

In [ ]:
# 8. Build the report (reads artifacts only; never retrains).
!python scripts/build_density_report.py --campaign-dir artifacts/density_lab/smoke_v0
from IPython.display import Image, display
for name in ('quality_by_target', 'rare_mode_recovery', 'feature_view_comparison', 'capacity_frontier'):
    display(Image('artifacts/density_lab/smoke_v0/report/%s.png' % name))